# Quickstart: Call the workshop models

Use one team key to call chat, embeddings, and image generation. No cloud account or local GPU is required.

**Before you start:** complete [setup](../docs/00_prework.md). Allow about 10 minutes for all three operations.

**Live session:** run Step 1 and Step 3 only. The final validation cell checks all three operations, so save it for after you complete the full notebook.

| Operation | Path | Model alias |
|---|---|---|
| Chat | `POST /v1/chat/completions` | `hackathon-chat` |
| Embeddings | `POST /v1/embeddings` | `hackathon-embedding` |
| Images | `POST /v1/images/generations` | `hackathon-image` |

[Notebook index](README.md) · [API reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md#inference)

These examples use the workshop service configuration. Confirm aliases and quotas with the organizers. Event details used as sample data are fictional.


## Step 1: Set up your environment

In [1]:
# Install the packages used in this notebook.
%pip install -q "openai>=3,<4" python-dotenv requests


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Run Jupyter from the workshop directory or its notebooks directory.
workshop_dir = Path.cwd().resolve()
if workshop_dir.name == "notebooks":
    workshop_dir = workshop_dir.parent
env_file = workshop_dir / ".env"
if not (workshop_dir / ".env.example").is_file() or not env_file.is_file():
    raise RuntimeError("Open Jupyter from the workshop directory after completing docs/00_prework.md.")
load_dotenv(env_file, override=True)

base_url = os.environ.get("OPENAI_BASE_URL", "").strip().rstrip("/")
api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not base_url or not api_key or api_key == "hk_paste_your_team_key_here":
    raise RuntimeError("Set OPENAI_BASE_URL and your team key in the workshop .env file.")
print("Endpoint:", base_url)
print("Team key loaded. Key value is hidden.")
client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)


Endpoint: https://ws.akamai-workshops.com/v1
Team key loaded. Key value is hidden.


## Step 2: Inspect an HTTP request

This optional example sends a request directly with `requests`. Inspect the status, rate-limit headers, and JSON response. Every request uses the configured base URL and `Authorization: Bearer <team-key>`.

The SDK examples below send the same types of HTTP requests and return Python objects.

In [3]:
import json
import requests

url = f"{base_url}/chat/completions"
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
body = {"model": "hackathon-chat", "messages": [{"role": "user", "content": "Say hello."}], "max_tokens": 20}

r = requests.post(url, headers=headers, json=body, timeout=60)
print("Status:", r.status_code)
print("Rate-limit headers:", {k: v for k, v in r.headers.items() if k.lower().startswith("x-ratelimit")})
print(json.dumps(r.json(), indent=2)[:900])

Status: 200
Rate-limit headers: {'x-ratelimit-limit': '100000', 'x-ratelimit-remaining': '100000', 'x-ratelimit-reset': '44'}
{
  "model": "Qwen/Qwen3.8-27B-FP8",
  "usage": {
    "prompt_tokens": 15,
    "completion_tokens": 10,
    "total_tokens": 25,
    "completion_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "choices": [
    {
      "message": {
        "content": "Hello! How can I help you today?",
        "role": "assistant",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "reasoning": null
      },
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": null
    }
  ],
  "id": "chatcmpl-818b912cf2508363",
  "object": "chat.completion",
  "created": 1790956996,
  "system_fingerprint": "vllm-0.30.0-tp4-34cf4502",
  "prompt_logprobs": null,
  "prompt_token_ids": null,
  "prompt_text": null,
  "kv_transfer_pa

Continue with the OpenAI Python client. The base URL selects the workshop service; the team key authenticates your requests.

## Step 3: Chat

Send `model`, `messages`, and an output budget with `max_tokens`. Read the answer from `choices[0].message.content` and check `finish_reason` for truncation.

Run the cell, change the user message to a question relevant to your project, and run it again. This is the participant exercise for the live workshop.

In [4]:
response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Keep answers short."},
        {"role": "user", "content": "What is an API key?"},
    ],
    max_tokens=80,
    temperature=0,
)
print(response.choices[0].message.content)
print("\nfinish_reason:", response.choices[0].finish_reason, "| tokens:", response.usage.total_tokens)

An API key is a unique string of characters used to authenticate a user or application when making requests to an API. It acts like a password, verifying your identity and allowing the service to track usage, enforce limits, and secure access.

finish_reason: stop | tokens: 81


Optional: set `stream=True` to receive text incrementally. Each chunk may contain `delta.content`; some chunks contain only metadata.

In [5]:
for chunk in client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Count from one to five, one number per line."}],
    max_tokens=40,
    stream=True,
):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

1
2
3
4
5


## Step 4: Embeddings

Send one string or up to 16 strings with `input`. Each response item contains an embedding. The workshop configuration uses 1,024 dimensions.

The example compares two similar sentences with an unrelated sentence. Similarity is a ranking signal, not a probability that a statement is correct.

In [6]:
response = client.embeddings.create(
    model="hackathon-embedding",
    input=["The cat sat on the mat.", "A feline rested on a rug.", "Quarterly revenue grew 8 percent."],
)
vectors = [item.embedding for item in response.data]
print(len(vectors), "vectors, each", len(vectors[0]), "numbers")

# Similar meaning gives similar vectors. Compare the first text to the other two.
import math
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))
print("cat vs feline:  ", round(cosine(vectors[0], vectors[1]), 3))
print("cat vs revenue: ", round(cosine(vectors[0], vectors[2]), 3))

3 vectors, each 1024 numbers
cat vs feline:   0.709
cat vs revenue:  0.274


## Step 5: Images

Send a text `prompt`, a supported square `size`, and `response_format="b64_json"`. The response contains one base64-encoded PNG. This example saves it as `quickstart.png` in the kernel's working directory.

In [ ]:
import base64
from IPython.display import Image as ShowImage

response = client.images.generate(
    model="hackathon-image",
    prompt="A yellow bicycle leaning on a brick wall, simple illustration",
    size="512x512",
    response_format="b64_json",
    extra_body={"seed": 3},
)
png_bytes = base64.b64decode(response.data[0].b64_json)
with open("quickstart.png", "wb") as f:
    f.write(png_bytes)
print("Saved quickstart.png,", len(png_bytes), "bytes")
ShowImage(data=png_bytes)

## Step 6: Handle errors

The team shares the deployment's quotas across notebooks and applications. Current workshop values are documented in the [API reference](../docs/inference.md#deployment-limits-to-confirm).

| Result | Action |
|---|---|
| `401` | Check the team key and endpoint |
| `404` | Check the supported path and alias |
| `422` | Check request fields and size limits |
| `429` | Respect retry headers and use bounded backoff |
| `finish_reason: length` | Reduce input or raise the output budget within the limit |

## Validation

Run this only after completing chat, embeddings, and images. It checks the embedding dimension and that image data was returned. Inspect the chat answer separately.

In [ ]:
assert response.data[0].b64_json and len(vectors[0]) == 1024
print("Quickstart complete. Your key, the endpoint, and all three operations work.")

## Clean up

The next cell removes the image created by this notebook. No resources were provisioned in your cloud account.

In [ ]:
from pathlib import Path
Path("quickstart.png").unlink(missing_ok=True)
print("Removed quickstart.png")

## Continue with your project

Choose [chat](01_chat_completions.ipynb), [retrieval](02_embeddings_and_retrieval.ipynb), [images](03_image_generation.ipynb), or [tools and JSON](04_tools_and_structured_output.ipynb).

Then choose a cloud component in [Build your project](../docs/build.md). Return to the [live workshop](../docs/workshop.md) if you are following the session.